# 🔬 Self-Supervised Denoising of Low-Dose Electron Microscopy Images

---

## Overview

Beam-sensitive biological and radiation-sensitive materials specimens force
EM operators into a **dose-fidelity trade-off**: enough electrons to see
structure, but few enough to avoid destroying it before the image is
recorded. Low electron dose means severe **shot noise**, and classical
denoisers either blur real structure or need clean/noisy image pairs that
do not exist for a single irradiated specimen.

This notebook builds a **self-supervised, blind-spot denoising pipeline**
(the Noise2Void family) that trains directly on noisy EM images — no clean
targets required — and validates it on a **real published EM dataset**:

| Module | Topic |
|--------|-------|
| **1**  | Real EM Data & Low-Dose Noise Model (Poisson shot noise + Gaussian read noise) |
| **2**  | Blind-Spot Self-Supervision: J-Invariance & the Masking Scheme |
| **3**  | Architecture & Masked Training Loop |
| **4**  | Quantitative Evaluation: PSNR / SSIM vs. Dose |
| **5**  | Production Methods & Limitations |

### Dataset

We use the **ISBI 2012 EM Segmentation Challenge** stack (Cardona et al.,
*PLoS Biology* 2010): serial-section transmission EM (ssTEM) of the
*Drosophila melanogaster* first-instar larva ventral nerve cord,
1024×1024 px, mirrored on GitHub. These are real, already-clean
publication-quality EM images — we synthetically re-degrade them to
low-dose conditions so we have a trustworthy pseudo-ground-truth for
**benchmarking** (the training procedure itself never sees this clean
reference; only degraded images are used for learning).

### The Core Imaging Equation

$$I_{\text{low-dose}}(\mathbf{x}) = \text{Poisson}\!\left(\frac{\rho(\mathbf{x})}{q}\right)\cdot q \;+\; \eta_{\text{read}}(\mathbf{x}), \qquad \eta_{\text{read}} \sim \mathcal{N}(0,\sigma_r^2)$$

where $q$ is the **dose scale** (smaller $q$ = fewer electrons = more
relative shot noise) and $\sigma_r$ is detector read noise.

> **Prerequisites:** `numpy`, `scipy`, `torch`, `scikit-image`, `tifffile`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# torch, tifffile imported at top-level to avoid NameError
# if cells are run out of order.
# ============================================================
import os
import glob
import warnings
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tifffile
import torch
import torch.nn as nn
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

warnings.filterwarnings("ignore")
np.random.seed(0)
torch.manual_seed(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — DOWNLOAD REAL EM DATA (ISBI 2012 / Cardona et al. 2010)
# ============================================================
DATA_DIR = "em_data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = ("https://raw.githubusercontent.com/unidesigner/"
            "groundtruth-drosophila-vnc/master/stack1/raw/{:02d}.tif")

N_SLICES = 20  # slices 00-19 are available in this mirror
for i in range(N_SLICES):
    fpath = os.path.join(DATA_DIR, f"slice{i:02d}.tif")
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(BASE_URL.format(i), fpath)
        except Exception as e:
            print(f"  Warning: could not fetch slice {i:02d}: {e}")

slice_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
print(f"Loaded {len(slice_files)} real ssTEM slices "
      f"(Cardona et al. 2010, Drosophila VNC)")

em_slices = [tifffile.imread(f).astype(np.float32) / 255.0 for f in slice_files]
print("Slice shape:", em_slices[0].shape, "| value range: [0, 1] after normalization")

## 1.1 Why Poisson Statistics Dominate at Low Dose

Electron detection is a **counting process**: each pixel accumulates a
number of electron (or photon, post-scintillator) events whose variance
equals its mean (Poisson statistics). At normal dose, thousands of
electrons per pixel make the *relative* fluctuation
$\sigma/\mu = 1/\sqrt{N}$ negligible. Cutting the dose by a factor of
$k$ cuts $N$ by $k$ and raises the relative noise by $\sqrt{k}$ — this
is why low-dose cryo-EM and beam-sensitive materials imaging are
fundamentally noise-limited, not resolution-limited, at the single-image
level.

We model the detector as Poisson shot noise (dose-dependent) plus a
small dose-independent Gaussian **read noise** floor from the detector
electronics.

In [ ]:
# ============================================================
# MODULE 1 — LOW-DOSE NOISE SIMULATION
# ============================================================
def simulate_low_dose(clean, dose_scale=0.06, read_noise_std=0.02, rng=None):
    """
    Poisson-Gaussian low-dose degradation.

    dose_scale: smaller -> fewer effective electrons -> more shot noise.
                (equivalent electron count per pixel ~ 1/dose_scale)
    read_noise_std: dose-independent Gaussian detector noise floor.
    """
    rng = rng or np.random
    photon_equiv = np.clip(clean, 0, None) / dose_scale
    shot = rng.poisson(photon_equiv).astype(np.float32) * dose_scale
    noisy = shot + rng.normal(0, read_noise_std, size=clean.shape).astype(np.float32)
    return np.clip(noisy, 0, 1)


def random_patch(img, size=128, rng=None):
    rng = rng or np.random
    h, w = img.shape
    y, x = rng.randint(0, h - size), rng.randint(0, w - size)
    return img[y:y + size, x:x + size]


# ------------------------------------------------------------------
# VISUALIZATION 1 — Dose sweep on a real EM slice
# ------------------------------------------------------------------
demo_clean = random_patch(em_slices[0], size=256)
dose_levels = [1.0, 0.15, 0.06, 0.02]  # 1.0 = (near) full dose passthrough

fig, axes = plt.subplots(1, len(dose_levels), figsize=(16, 4.5))
fig.suptitle("Module 1 — Low-Dose Degradation of a Real EM Image\n"
             "(Cardona et al. 2010, Drosophila ventral nerve cord, ssTEM)",
             color=ACCENT, fontweight="bold")
for ax, dose in zip(axes, dose_levels):
    if dose >= 1.0:
        deg = demo_clean
        title = "Full dose\n(reference)"
    else:
        deg = simulate_low_dose(demo_clean, dose_scale=dose)
        p = psnr_metric(demo_clean, deg, data_range=1.0)
        title = f"dose_scale={dose}\nPSNR={p:.1f} dB"
    ax.imshow(deg, cmap="gray")
    ax.set_title(title, color=TEXT, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 2 — Blind-Spot Self-Supervision

## 2.1 The Core Problem With Ordinary Self-Supervised Denoising

A plain CNN trained to reconstruct its own noisy input with an
identity-like objective ($f(I) \approx I$) trivially learns the identity
function — it just copies the noisy pixel through, since that is the
global minimum of $\mathbb{E}\|f(I)-I\|^2$ when $f$ can see the pixel
it is asked to predict.

## 2.2 J-Invariance and the Noise2Void Trick

**Noise2Void** (Krull et al., *CVPR* 2019) breaks this trivial solution
by making the network **blind to the pixel it must predict**:

1. Pick a random small set of pixels in the noisy image (the *masked set* $J$)
2. Replace each masked pixel's value with a value copied from a nearby
   random neighbor (never its own value)
3. Train the network to predict the **original noisy value** at those
   masked locations from the corrupted input — the network never sees the
   ground truth it must predict, only context around it

This makes the network **$J$-invariant**: its output at a masked pixel
does not depend on the input value at that pixel, only on its
surroundings. Because pixel-wise noise is (by assumption) conditionally
independent given the true underlying signal, the only way to predict a
masked noisy pixel well from its *neighbors alone* is to predict the
**true signal**, not the noise realization. This is why a single noisy
image is enough — no clean targets or paired noisy realizations needed
(contrast with **Noise2Noise**, Lehtinen et al. 2018, which needs two
independent noisy captures of the same scene).

## 2.3 Masking Fraction

Only a small fraction (~2%) of pixels are masked per training patch, so
each gradient step still has abundant unmasked context; over many
iterations every pixel is masked many times.

In [ ]:
# ============================================================
# MODULE 2 — BLIND-SPOT MASK GENERATION
# ============================================================
def generate_blind_spot_mask(shape, frac=0.02, rng=None):
    """Randomly select ~frac of pixels to be blind-spot masked."""
    rng = rng or np.random
    mask = np.zeros(shape, dtype=bool)
    n = int(frac * shape[0] * shape[1])
    ys = rng.randint(0, shape[0], n)
    xs = rng.randint(0, shape[1], n)
    mask[ys, xs] = True
    return mask


def apply_blind_spot(img, mask, radius=2, rng=None):
    """
    Replace each masked pixel with a value copied from a random
    neighbor within `radius` (never the pixel's own value) — the
    standard Noise2Void UPS (uniform pixel selection) scheme.
    """
    rng = rng or np.random
    out = img.copy()
    h, w = img.shape
    ys, xs = np.where(mask)
    dy = rng.randint(-radius, radius + 1, size=len(ys))
    dx = rng.randint(-radius, radius + 1, size=len(ys))
    ny = np.clip(ys + dy, 0, h - 1)
    nx = np.clip(xs + dx, 0, w - 1)
    out[ys, xs] = img[ny, nx]
    return out


# ------------------------------------------------------------------
# VISUALIZATION 2 — What the network actually sees during training
# ------------------------------------------------------------------
demo_noisy = simulate_low_dose(demo_clean, dose_scale=0.06)
demo_mask = generate_blind_spot_mask(demo_noisy.shape, frac=0.02)
demo_masked_input = apply_blind_spot(demo_noisy, demo_mask)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle("Module 2 — Blind-Spot Masking Scheme (Noise2Void)",
             color=ACCENT, fontweight="bold")
axes[0].imshow(demo_noisy, cmap="gray")
axes[0].set_title("Noisy input (training target\nat masked locations)", color=TEXT, fontsize=10)
axes[1].imshow(demo_mask, cmap="Reds")
axes[1].set_title(f"Blind-spot mask\n({demo_mask.mean()*100:.1f}% of pixels)", color=TEXT, fontsize=10)
axes[2].imshow(demo_masked_input, cmap="gray")
axes[2].set_title("Network input\n(masked px replaced by neighbor)", color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 3 — Architecture & Masked Training

## 3.1 Why an Ordinary U-Net Works Here

Unlike some blind-spot variants that need a *structurally* constrained
receptive field (e.g. shifted convolutions), Noise2Void enforces
$J$-invariance entirely through the **masking procedure**, so a compact,
ordinary convolutional U-Net is sufficient — the loss is only ever
computed at masked pixels, whose true value never appears in the input.

## 3.2 Masked Loss

$$\mathcal{L} = \frac{1}{|J|}\sum_{i \in J}\bigl(f(I_{\text{masked}})_i - I_i\bigr)^2$$

only the masked pixel set $J$ contributes to the gradient each step.

In [ ]:
# ============================================================
# MODULE 3 — TINY U-NET + MASKED SELF-SUPERVISED TRAINING LOOP
# ============================================================
class TinyUNet(nn.Module):
    """Compact 2-level U-Net; no blind-spot architectural constraint needed —
    J-invariance comes entirely from the masking scheme (Module 2)."""

    def __init__(self, base_ch=32):
        super().__init__()
        c = base_ch
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1), nn.ReLU(inplace=True))
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(c, c * 2, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(c * 2, c * 2, 3, padding=1), nn.ReLU(inplace=True))
        self.up = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(c * 2, c, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1), nn.ReLU(inplace=True))
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        d1 = self.dec1(torch.cat([self.up(e2), e1], dim=1))
        return self.out(d1)


def train_step(model, opt, patch_pool, dose_scale, patch_size=128, mask_frac=0.02):
    src = patch_pool[np.random.randint(len(patch_pool))]
    clean_patch = random_patch(src, size=patch_size)
    noisy = simulate_low_dose(clean_patch, dose_scale=dose_scale)
    mask = generate_blind_spot_mask(noisy.shape, frac=mask_frac)
    masked_in = apply_blind_spot(noisy, mask)

    x = torch.from_numpy(masked_in).float()[None, None].to(DEVICE)
    target = torch.from_numpy(noisy).float()[None, None].to(DEVICE)
    m = torch.from_numpy(mask).float()[None, None].to(DEVICE)

    pred = model(x)
    loss = ((pred - target) ** 2 * m).sum() / m.sum().clamp_min(1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss.item()


DOSE_SCALE = 0.06        # ~ equivalent to a moderately aggressive low-dose regime
N_STEPS = 600
train_slices = em_slices[:16]   # held-out slices reserved for evaluation (Module 4)
eval_slices = em_slices[16:]

model = TinyUNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loss_history = []
for step in range(N_STEPS):
    loss_history.append(train_step(model, optimizer, train_slices, DOSE_SCALE))
    if (step + 1) % 150 == 0:
        print(f"step {step+1:4d}/{N_STEPS} | masked-pixel MSE = {np.mean(loss_history[-150:]):.5f}")

print("Training complete.")

In [ ]:
# ------------------------------------------------------------------
# VISUALIZATION 3 — Training loss curve
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
smoothed = np.convolve(loss_history, np.ones(20) / 20, mode="valid")
ax.plot(loss_history, color="#3fb950", alpha=0.3, label="per-step loss")
ax.plot(range(19, 19 + len(smoothed)), smoothed, color=ACCENT, lw=2, label="20-step moving avg")
ax.set_xlabel("Training step"); ax.set_ylabel("Masked-pixel MSE")
ax.set_title("Module 3 — Self-Supervised Training Loss (no clean labels used)",
             color=ACCENT, fontweight="bold")
ax.legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.tight_layout()
plt.show()

---
# Module 4 — Quantitative Evaluation

## 4.1 The Self-Supervised Evaluation Paradox

In a genuine low-dose acquisition there **is no clean reference** — that
is precisely the problem being solved. Here, because our "noisy" images
were *synthetically* degraded from real, already-clean published EM
data, we retain the original as a **pseudo-ground-truth for
benchmarking only**. The training procedure itself (Modules 2-3) never
sees or uses this clean reference — this evaluation step is purely to
validate the method, exactly mirroring how Noise2Void/Topaz-Denoise
papers validate against held-out synthetic or simulated degradations
before deploying on real unlabeled dose-limited data.

## 4.2 Metrics

- **PSNR** (dB): pixel-fidelity, sensitive to global error magnitude
- **SSIM**: structural similarity, more sensitive to perceptually
  relevant local contrast and edges — often more informative for
  judging whether fine EM structures (membranes, vesicles) are preserved

In [ ]:
# ============================================================
# MODULE 4 — EVALUATION ON HELD-OUT SLICES
# ============================================================
model.eval()


def denoise(model, img):
    with torch.no_grad():
        x = torch.from_numpy(img).float()[None, None].to(DEVICE)
        return model(x)[0, 0].cpu().numpy()


rng_eval = np.random.RandomState(42)
n_eval_patches = 12
results = {"noisy_psnr": [], "denoised_psnr": [], "noisy_ssim": [], "denoised_ssim": []}
example_panels = []

for i in range(n_eval_patches):
    src = eval_slices[i % len(eval_slices)]
    clean = random_patch(src, size=128, rng=rng_eval)
    noisy = simulate_low_dose(clean, dose_scale=DOSE_SCALE, rng=rng_eval)
    den = denoise(model, noisy)

    results["noisy_psnr"].append(psnr_metric(clean, noisy, data_range=1.0))
    results["denoised_psnr"].append(psnr_metric(clean, den, data_range=1.0))
    results["noisy_ssim"].append(ssim_metric(clean, noisy, data_range=1.0))
    results["denoised_ssim"].append(ssim_metric(clean, den, data_range=1.0))
    if i < 3:
        example_panels.append((clean, noisy, den))

print(f"Mean PSNR   — noisy: {np.mean(results['noisy_psnr']):.2f} dB   "
      f"| denoised: {np.mean(results['denoised_psnr']):.2f} dB   "
      f"(Δ = +{np.mean(results['denoised_psnr'])-np.mean(results['noisy_psnr']):.2f} dB)")
print(f"Mean SSIM   — noisy: {np.mean(results['noisy_ssim']):.3f}   "
      f"| denoised: {np.mean(results['denoised_ssim']):.3f}")

# ------------------------------------------------------------------
# VISUALIZATION 4 — Example patches + metric distributions
# ------------------------------------------------------------------
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Module 4 — Denoising Results on Held-Out Real EM Patches",
             color=ACCENT, fontweight="bold", fontsize=14)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.3)

col_titles = ["Clean (pseudo-GT)", "Low-dose noisy input", "Blind-spot denoised", "Abs. error (denoised)"]
for row, (clean, noisy, den) in enumerate(example_panels):
    err = np.abs(clean - den)
    for col, (im, cmap, vmax) in enumerate(zip(
            [clean, noisy, den, err], ["gray", "gray", "gray", "inferno"], [1, 1, 1, 0.3])):
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(im, cmap=cmap, vmin=0, vmax=vmax)
        if row == 0:
            ax.set_title(col_titles[col], color=TEXT, fontsize=10)
        ax.axis("off")

ax_bar = fig.add_subplot(gs[2, 3])
ax_bar.axis("off")
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4))
for ax, key_pair, ylabel in zip(
        axes2, [("noisy_psnr", "denoised_psnr"), ("noisy_ssim", "denoised_ssim")], ["PSNR (dB)", "SSIM"]):
    ax.boxplot([results[key_pair[0]], results[key_pair[1]]], labels=["Noisy", "Denoised"],
               patch_artist=True,
               boxprops=dict(facecolor="#21262d", color=TEXT),
               medianprops=dict(color=ACCENT), whiskerprops=dict(color=TEXT), capprops=dict(color=TEXT))
    ax.set_ylabel(ylabel)
fig2.suptitle(f"Metric Distributions over {n_eval_patches} Held-Out Patches", color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Training Requirement | Notes |
|--------|----------------------|-------|
| Noise2Noise (Lehtinen 2018) | Paired independent noisy captures | Needs 2 acquisitions of same field; extra dose |
| Noise2Void (Krull 2019) | Single noisy image, blind-spot masking | Used in this notebook |
| Noise2Self (Batson & Royer 2019) | Single noisy image, $J$-invariant partitions | Generalizes N2V's masking theory |
| Topaz-Denoise (Bepler et al. 2020) | Pre-trained on large cryo-EM corpora | Deployed U-Net trained across many micrographs |
| cryoCARE (Buchholz et al. 2019) | Paired odd/even frame sums | Content-aware restoration for cryo-ET tilt series |

## Known Limitations of This Tutorial
- Blind-spot masking assumes **pixel-independent noise**; correlated
  detector noise (e.g. from binning or drift correction) violates this
  and reduces denoising quality.
- A single dose level was trained per model here; production tools
  (Topaz-Denoise) train across a distribution of doses/defoci for
  robustness.
- Evaluation used *synthetically* re-degraded already-clean EM images as
  pseudo-ground-truth; genuine low-dose acquisitions require perceptual
  or downstream-task metrics (e.g. segmentation accuracy after
  denoising) since no clean reference exists.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Self-Supervised Low-Dose EM Denoising — Pipeline Dashboard",
             fontsize=16, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(demo_clean, cmap="gray"); ax0.set_title("Real EM slice (source)", color=TEXT, fontsize=10); ax0.axis("off")
ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(demo_noisy, cmap="gray"); ax1.set_title(f"Low-dose sim. (dose_scale={DOSE_SCALE})", color=TEXT, fontsize=10); ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(demo_mask, cmap="Reds"); ax2.set_title("Blind-spot mask", color=TEXT, fontsize=10); ax2.axis("off")
den_demo = denoise(model, demo_noisy)
ax3 = fig.add_subplot(gs[0, 3]); ax3.imshow(den_demo, cmap="gray"); ax3.set_title("Denoised output", color=TEXT, fontsize=10); ax3.axis("off")

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(loss_history, color="#3fb950", alpha=0.3)
ax4.plot(range(19, 19 + len(smoothed)), smoothed, color=ACCENT, lw=2)
ax4.set_title("Training loss (masked-pixel MSE)", color=TEXT, fontsize=10)
ax4.set_xlabel("step")

ax5 = fig.add_subplot(gs[1, 2])
ax5.bar(["Noisy", "Denoised"], [np.mean(results["noisy_psnr"]), np.mean(results["denoised_psnr"])],
        color=["#f0883e", ACCENT])
ax5.set_title("Mean PSNR (dB)", color=TEXT, fontsize=10)

ax6 = fig.add_subplot(gs[1, 3])
ax6.bar(["Noisy", "Denoised"], [np.mean(results["noisy_ssim"]), np.mean(results["denoised_ssim"])],
        color=["#f0883e", ACCENT])
ax6.set_title("Mean SSIM", color=TEXT, fontsize=10)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Real data acquisition | 1 | Downloaded genuine published ssTEM data (Cardona et al. 2010) rather than a synthetic phantom |
| Noise modeling | 1 | Poisson shot noise dominates at low dose; read noise is a small dose-independent floor |
| Blind-spot masking | 2 | $J$-invariance breaks the trivial identity-copy solution without needing clean targets |
| Architecture & training | 3 | Ordinary U-Net; masked MSE loss computed only at masked pixels |
| Evaluation | 4 | PSNR/SSIM gains measured against a pseudo-ground-truth reserved only for benchmarking |
| Context | 5 | Positioned against Noise2Noise, Noise2Self, Topaz-Denoise, cryoCARE |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Poisson noise simulation | $\mathcal{O}(N^2)$ | Negligible |
| Blind-spot mask + apply | $\mathcal{O}(\text{frac}\cdot N^2)$ | Negligible |
| U-Net forward/backward | $\mathcal{O}(N^2 \cdot C)$ | Dominant; GPU-parallelisable |
| PSNR/SSIM evaluation | $\mathcal{O}(N^2)$ | Negligible |

## Key References
- Cardona et al. (2010) — ssTEM Drosophila VNC dataset (*PLoS Biology*)
- Lehtinen et al. (2018) — Noise2Noise (*ICML*)
- Krull, Buchholz & Jug (2019) — Noise2Void (*CVPR*)
- Batson & Royer (2019) — Noise2Self (*ICML*)
- Bepler et al. (2020) — Topaz-Denoise (*Nature Communications*)
- Buchholz et al. (2019) — cryoCARE (*ICCV Workshops*)